In [1]:
import numpy as np
from plotly.io import show

from skfolio import Population
from skfolio.datasets import load_sp500_dataset
from skfolio.model_selection import WalkForward, cross_val_predict
from skfolio.optimization import MeanRisk, ObjectiveFunction
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()
prices = prices[["AAPL", "GE", "JPM"]]

X = prices_to_returns(prices)

In [2]:
model = MeanRisk(objective_function=ObjectiveFunction.MAXIMIZE_UTILITY)
model.fit(X)
model.weights_

array([6.17733231e-01, 3.78774141e-09, 3.82266765e-01])

In [3]:
management_fees = {"AAPL": 0.03 / 252, "GE": 0.06 / 252, "JPM": 0.01 / 252}
# Same as management_fees = np.array([0.03, 0.06, 0.01]) / 252

model_mf = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_UTILITY,
    management_fees=management_fees,
)
model_mf.fit(X)
model_mf.weights_

array([5.74787861e-01, 1.43028108e-08, 4.25212125e-01])

In [4]:
model_mf.weights_ - model.weights_

array([-4.29453703e-02,  1.05150694e-08,  4.29453598e-02])

In [5]:
#multiperiod porfolio
holding_period = 60
fitting_period = 60
cv = WalkForward(train_size=fitting_period, test_size=holding_period)

In [6]:
management_fees = np.array([0.03, 0.06, 0.01]) / 252

In [7]:
model = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_UTILITY,
    portfolio_params=dict(management_fees=management_fees),
)
# pred1 is a MultiPeriodPortfolio
pred1 = cross_val_predict(model, X, cv=cv, n_jobs=-1)
pred1.name = "pred1"

In [8]:
model.set_params(management_fees=management_fees)
pred2 = cross_val_predict(model, X, cv=cv, n_jobs=-1)
pred2.name = "pred2"

In [9]:
population = Population([pred1, pred2])
fig = population.plot_cumulative_returns()
show(fig)

We notice that the model fitted with MF outperform the model fitted without MF.